# 00 Everything in one GPU session
Embeddings -> (heads on CPU in the background) -> generative VLM -> CNN baselines -> CNN + VLM pseudo-labels -> final comparisons. Each stage is resumable and records its status in results/00_stage_status.json; a failing stage does not stop the others.

In [ ]:
!pip -q install transformers==4.51.3 open_clip_torch==2.32.0 sentencepiece

In [ ]:
import glob, os, sys
if os.path.isdir("/kaggle/input"):
    hits = glob.glob("/kaggle/input/**/src/woundvlm/__init__.py", recursive=True)
    assert hits, "attach the project dataset (lowerlimbwoundvlm) as an input"
    SRC = hits[0].rsplit("/woundvlm/", 1)[0]
else:
    SRC = os.path.abspath("../src")
sys.path.insert(0, SRC)
from woundvlm import env
E = env.setup()
P = E["project"]
print(E)

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda" or not E["in_kaggle"], "select a GPU accelerator (T4) for this notebook" 

In [ ]:
import gc, json, subprocess, time, traceback
STATUS = os.path.join(E["results"], "00_stage_status.json")
status = json.load(open(STATUS)) if os.path.exists(STATUS) else {}

def stage(name, fn):
    t = time.time()
    print(f"===== {name} =====", flush=True)
    try:
        fn(); status[name] = {"ok": True}
    except Exception as e:
        traceback.print_exc(); status[name] = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    status[name]["minutes"] = round((time.time() - t) / 60, 1)
    json.dump(status, open(STATUS, "w"), indent=1)
    gc.collect(); torch.cuda.empty_cache()
    print(name, status[name], flush=True)

from woundvlm import experiments, report

In [ ]:
stage("embeddings", lambda: experiments.run_embeddings(P, device=DEVICE))

In [ ]:
# CPU heads grid in a background process while the GPU trains (needs the SigLIP embeddings)
heads_log = open(os.path.join(E["results"], "03_heads.log"), "w")
code = f"import sys; sys.path.insert(0, {SRC!r}); from woundvlm import experiments; experiments.run_heads({P!r}, n_jobs=2)"
heads_proc = subprocess.Popen([sys.executable, "-c", code], stdout=heads_log, stderr=subprocess.STDOUT)
t_heads = time.time()
print("heads started, pid", heads_proc.pid)

In [ ]:
stage("generative_vlm", lambda: experiments.run_genvlm(P, device=DEVICE))

In [ ]:
stage("cnn_main", lambda: experiments.run_cnn(P, "main", device=DEVICE))

In [ ]:
rc = heads_proc.wait(); heads_log.close()
status["heads"] = {"ok": rc == 0, "returncode": rc, "minutes": round((time.time() - t_heads) / 60, 1)}
json.dump(status, open(STATUS, "w"), indent=1)
print(status["heads"]); print(open(os.path.join(E["results"], "03_heads.log")).read()[-3000:])

In [ ]:
stage("cnn_pseudo", lambda: experiments.run_cnn(P, "pseudo", device=DEVICE))

In [ ]:
stage("report", lambda: print(report.write_all(P)))

In [ ]:
print(json.dumps(status, indent=1))